In [1]:
!pip install langchain langchain-community langsmith transformers accelerate -q

In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "APIKEY"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening-system"

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.llms import HuggingFacePipeline

from transformers import pipeline
import json, re

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.llms import HuggingFacePipeline

from transformers import pipeline
import json, re

In [5]:
job_description = """
Looking for a Data Scientist with:
- Python, Machine Learning
- NLP, Deep Learning
- Pandas, NumPy, Scikit-learn
- Experience with Transformers and LLMs
- 2+ years experience
"""

In [6]:
strong_resume = """
Name: Rahul Sharma

Experience:
3.5 years as Data Scientist at Infosys

Skills:
Python, Machine Learning, Deep Learning, Natural Language Processing (NLP),
Transformers, Large Language Models (LLMs)

Tools:
Pandas, NumPy, Scikit-learn, TensorFlow, PyTorch, Hugging Face, SQL

Projects:
- Built NLP chatbot using transformers
- Developed recommendation system using ML
- Worked on sentiment analysis using deep learning

Education:
B.Tech in Computer Science

Summary:
Strong experience in ML, NLP and production-level AI systems.
"""

average_resume = """
Name: Priya Verma

Experience:
1.2 years as Data Analyst

Skills:
Python, Data Analysis, Basic Machine Learning

Tools:
Pandas, NumPy, Excel, Matplotlib

Projects:
- Sales data analysis dashboard
- Basic ML model for prediction

Education:
B.Sc in Statistics

Summary:
Good in data analysis but limited exposure to NLP and deep learning.
"""


weak_resume = """
Name: Amit Kumar

Experience:
Fresher

Skills:
Excel, Communication, MS Word

Tools:
Excel, PowerPoint

Projects:
- College presentation project

Education:
B.Com

Summary:
No experience in programming or machine learning.
"""

In [7]:
hf_pipeline = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=200,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_24418/3430423543.py:8: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [15]:
job_prompt = PromptTemplate(
    input_variables=["job"],
    template="""
Extract job requirements.

Return JSON:
{{
 "skills": [],
 "tools": [],
 "min_experience": 0
}}

Job:
{job}
"""
)

In [16]:
resume_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract candidate info.

Return JSON:
{{
 "skills": [],
 "tools": [],
 "experience": ""
}}

Resume:
{resume}
"""
)

In [17]:
match_prompt = PromptTemplate(
    input_variables=["job", "candidate"],
    template="""
Compare candidate with job.

Return JSON:
{{
 "matched_skills": [],
 "missing_skills": []
}}

Job:
{job}

Candidate:
{candidate}
"""
)

In [18]:
explain_prompt = PromptTemplate(
    input_variables=["score", "breakdown"],
    template="""
Explain the score.

Score: {score}
Details: {breakdown}

Give short explanation.
"""
)

In [19]:
parser = StrOutputParser()

job_chain = job_prompt | llm | parser
resume_chain = resume_prompt | llm | parser
explain_chain = explain_prompt | llm | parser

In [20]:
def normalize(lst):
    return [str(x).lower().strip() for x in lst]

def extract_exp(text):
    nums = re.findall(r"\d+", str(text))
    return int(nums[0]) if nums else 0


def score_candidate(extracted, job):

    skills = normalize(extracted.get("skills", []))
    tools = normalize(extracted.get("tools", []))

    job_skills = normalize(job.get("skills", []))
    job_tools = normalize(job.get("tools", []))

    exp = extract_exp(extracted.get("experience", ""))
    req_exp = job.get("min_experience", 0)

    # skill score
    skill_score = len(set(skills) & set(job_skills)) / len(job_skills) * 100 if job_skills else 0

    # tool score
    tool_score = len(set(tools) & set(job_tools)) / len(job_tools) * 100 if job_tools else 0

    # experience score
    if exp >= req_exp:
        exp_score = 100
    elif exp == 0:
        exp_score = 0
    else:
        exp_score = (exp / req_exp) * 100

    final = 0.5*skill_score + 0.2*tool_score + 0.3*exp_score

    return round(final,2), {
        "skill_score": round(skill_score,2),
        "tool_score": round(tool_score,2),
        "experience_score": round(exp_score,2)
    }

In [24]:
import re, json

def clean_json(text):
    try:
        json_str = re.search(r"\{.*\}", text, re.DOTALL).group()
        return json.loads(json_str)
    except:
        return {}

In [25]:
def pipeline_run(resume):

    job_raw = job_chain.invoke({"job": job_description})
    job_data = clean_json(job_raw)

    resume_raw = resume_chain.invoke({"resume": resume})
    resume_data = clean_json(resume_raw)

    score, breakdown = score_candidate(resume_data, job_data)

    explanation = explain_chain.invoke({
        "score": score,
        "breakdown": breakdown
    })

    return {
        "Job": job_data,
        "Resume": resume_data,
        "Score": score,
        "Breakdown": breakdown,
        "Explanation": explanation
    }

In [26]:
print("STRONG\n")
print(pipeline_run(strong_resume))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STRONG



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'Job': {'skills': [], 'tools': [], 'min_experience': 0}, 'Resume': {'skills': [], 'tools': [], 'experience': ''}, 'Score': 30.0, 'Breakdown': {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}, 'Explanation': "\nExplain the score.\n\nScore: 30.0\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 0.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 0.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 0.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 0.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 0.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n"}


In [27]:
print("WEAK\n")
print(pipeline_run(weak_resume))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WEAK



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'Job': {'skills': [], 'tools': [], 'min_experience': 0}, 'Resume': {'skills': [], 'tools': [], 'experience': ''}, 'Score': 30.0, 'Breakdown': {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}, 'Explanation': "\nExplain the score.\n\nScore: 30.0\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n"}


In [28]:
print("AVERAGE\n")
print(pipeline_run(average_resume))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AVERAGE



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'Job': {'skills': [], 'tools': [], 'min_experience': 0}, 'Resume': {'skills': [], 'tools': [], 'experience': ''}, 'Score': 30.0, 'Breakdown': {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}, 'Explanation': "\nExplain the score.\n\nScore: 30.0\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nExample 1\n\nThe game provides a simple example of how the player's skill points can be spent on various features.\n\nWhat is the value of the following?\n\nPlayer's Skill Points\n\nIf you do not understand the value of the following, then you should definitely consider the following.\n\nThe value of the following is calculated with a simple formula.\n\nPlayer's Skill Points\n\nIn this example, the player is playing a passive role as he does not need to buy equipment.\n\nTo understand, let's take a look at the following formula:\n\nPlayer's Skill Points\n\nThe following formula is used to calculate the value of the following.\n\nPlaye

In [29]:
debug_resume = "Expert AI engineer with 10 years experience."

print("DEBUG\n")
print(pipeline_run(debug_resume))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'Job': {'skills': [], 'tools': [], 'min_experience': 0}, 'Resume': {'skills': [], 'tools': [], 'experience': ''}, 'Score': 30.0, 'Breakdown': {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}, 'Explanation': "\nExplain the score.\n\nScore: 30.0\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n\nScore: 30.0\n\nDetails: {'skill_score': 0, 'tool_score': 0, 'experience_score': 100}\n\nGive short explanation.\n"}


# Observations

- Strong candidate receives high score due to full skill match
- Average candidate lacks NLP and deep learning skills
- Weak candidate has no relevant skills

# Debug Insight
The model sometimes overestimates vague resumes.
This was observed in the debug case.

LangSmith tracing helps identify:
- Prompt issues
- Extraction mistakes
